<a href="https://colab.research.google.com/github/Jupiterian/ML-Learning-Portfolio/blob/main/Pass_Fail_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 30.4 MB/s eta 0:00:00


In [2]:
# Modules
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.optim import SGD
from torchmetrics import Accuracy
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd

In [3]:
# Device Agnostic Code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
# Load Data
data_path = "/content/Pass-Fail_Data.csv"
df = pd.read_csv(data_path)
df.head()

,student_id,attendance_pct,homework_pct,midterm_score,study_hours_per_week,pass
0,1,95,92,88,12,1
1,2,88,85,79,10,1
2,3,60,55,58,4,0
3,4,72,70,65,6,1
4,5,40,45,50,3,0


Important Note: always double check the order you have your test_train split set in.

In [5]:
# Preprocess Data
features = ["attendance_pct", "homework_pct", "midterm_score", "study_hours_per_week"]
labels = ["pass"]
X = torch.tensor(data=df[features].values, dtype=torch.float32)
y = torch.tensor(data=df[labels].values, dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [6]:
# Model Class
class BinaryModel(nn.Module):
    def __init__(self):
      super().__init__()
      self.layer1 = nn.Linear(in_features=4, out_features=5) # Corrected in_features to 4
      self.relu = nn.ReLU() # Assigned ReLU to self
      self.layer2 = nn.Linear(in_features=5, out_features=1)
    def forward(self, x):
      return self.layer2(self.relu(self.layer1(x))) # Applied ReLU

In [7]:
# Intiate model
torch.manual_seed(42)
model = BinaryModel()
model.to(device=device)

BinaryModel(
  (layer1): Linear(in_features=4, out_features=5, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=5, out_features=1, bias=True)
)

In [8]:
# Loss and optimizer functions
loss_fn = nn.BCEWithLogitsLoss() # Changed to BCEWithLogitsLoss for binary classification
optimizer = SGD(params=model.parameters(), lr=0.01)

In [10]:
# Training Loop

# Send training data to device
X_test = X_test.to(device)
X_train = X_train.to(device)
y_test = y_test.to(device)
y_train = y_train.to(device)

# epochs number
epochs = 101

for epoch in range(epochs):

  # 1. Put model in training mode
  model.train()

  # 2. Train the model
  y_pred = model(X_train)

  # 3. Loss function
  loss = loss_fn(y_pred, y_train)

  # 4. Optimizer Zero grad
  optimizer.zero_grad()

  # 5. Back gropagation
  loss.backward()

  # 6. Optimizer step
  optimizer.step()

  # Evaluate
  if epoch % 20 == 0:
    model.eval()
    y_pred_test = model(X_test)
    loss_test = loss_fn(y_pred_test, y_test)
    print(f"Epoch: {epoch}, Train Loss: {loss:.3f}, Test Loss: {loss_test:.3f}")

Epoch: 0, Train Loss: 0.584, Test Loss: 0.538
Epoch: 20, Train Loss: 0.581, Test Loss: 0.531
Epoch: 40, Train Loss: 0.576, Test Loss: 0.525
Epoch: 60, Train Loss: 0.574, Test Loss: 0.517
Epoch: 80, Train Loss: 0.568, Test Loss: 0.510
Epoch: 100, Train Loss: 0.563, Test Loss: 0.504


In [12]:
model.eval()
with torch.no_grad():
  test_pred_logits = model(X_test)
  # Convert logits to probabilities using sigmoid
  test_pred_probs = torch.sigmoid(test_pred_logits)
  # Threshold probabilities to get binary predicted labels (0 or 1)
  predicted_labels = (test_pred_probs >= 0.5).int()

  # Ensure y_test has the same shape as predicted_labels for accurate comparison
  # y_test is (N, 1), predicted_labels is (N, 1), so direct comparison is fine
  correct_predictions = (predicted_labels == y_test).sum().item()
  accuracy = correct_predictions / len(y_test) * 100
  print(f"Test accuracy: {accuracy:.2f}%")

Test accuracy: 96.67%
